**Imports and load data**

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dataset/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


**Create engineered features**

In [2]:
# Feature 1: Family size = siblings/spouses + parents/children + self
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature 2: IsAlone - whether the passenger was traveling solo
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


**Check these features actually relate to survival**

In [3]:
df.groupby('FamilySize')['Survived'].mean()

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64

In [4]:
df.groupby('IsAlone')['Survived'].mean()

IsAlone
0    0.505650
1    0.303538
Name: Survived, dtype: float64

**Prepare the feature set**

In [5]:
# Drop columns we won't use
df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Fill missing Age and Embarked (still needed even inside a pipeline for these two)
df_model['Age'] = df_model['Age'].fillna(df_model['Age'].median())
df_model['Embarked'] = df_model['Embarked'].fillna(df_model['Embarked'].mode()[0])

X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,male,22.0,1,0,7.2500,S,2,0
1,1,female,38.0,1,0,71.2833,C,2,0
2,3,female,26.0,0,0,7.9250,S,1,1
3,1,female,35.0,1,0,53.1000,S,2,0
4,3,male,35.0,0,0,8.0500,S,1,1


**Define which columns are numerical vs categorical**

In [6]:
numerical_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
categorical_cols = ['Pclass', 'Sex', 'Embarked', 'IsAlone']

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

Numerical: ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
Categorical: ['Pclass', 'Sex', 'Embarked', 'IsAlone']


**Split the data**

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Build the ColumnTransformer + Pipeline**

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

**Fit and evaluate the pipeline**

In [9]:
pipeline.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline.predict(X_test)

print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

Pipeline Accuracy: 0.7933
                 precision    recall  f1-score   support

Did Not Survive       0.80      0.86      0.83       105
       Survived       0.78      0.70      0.74        74

       accuracy                           0.79       179
      macro avg       0.79      0.78      0.78       179
   weighted avg       0.79      0.79      0.79       179



**Compare with/without engineered features**

In [10]:
numerical_cols_baseline = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_cols_baseline = ['Pclass', 'Sex', 'Embarked']

preprocessor_baseline = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols_baseline),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols_baseline)
    ]
)

pipeline_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor_baseline),
    ('classifier', LogisticRegression(max_iter=1000))
])

X_train_base = X_train[numerical_cols_baseline + categorical_cols_baseline]
X_test_base = X_test[numerical_cols_baseline + categorical_cols_baseline]

pipeline_baseline.fit(X_train_base, y_train)
y_pred_base = pipeline_baseline.predict(X_test_base)

print(f"Baseline (no engineered features) Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"With engineered features Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Baseline (no engineered features) Accuracy: 0.7989
With engineered features Accuracy: 0.7933


**Comparison table**

In [11]:
comparison = pd.DataFrame({
    'Version': ['Without FamilySize/IsAlone', 'With FamilySize/IsAlone'],
    'Accuracy': [accuracy_score(y_test, y_pred_base), accuracy_score(y_test, y_pred)]
})
comparison

,Version,Accuracy
0,Without FamilySize/IsAlone,0.798883
1,With FamilySize/IsAlone,0.793296


**Save the final pipeline**

In [12]:
import joblib

joblib.dump(pipeline, 'titanic_pipeline.pkl')
print("Pipeline saved as titanic_pipeline.pkl")

Pipeline saved as titanic_pipeline.pkl


**Quick sanity check that the saved file loads and works**

In [13]:
loaded_pipeline = joblib.load('titanic_pipeline.pkl')
test_prediction = loaded_pipeline.predict(X_test.iloc[:5])
print(test_prediction)

[0 0 0 1 1]
